# Finetune Notebook (SPR Probing)

"
"This notebook runs end-to-end finetuning with `probing.finetune` using the default mapping:
"
"- entailment: `label >= 4`
"
"- not entailment: `label <= 3`
"
"- drop inapplicable rows (`applicable == false`)

"
"Use `QUICK_RUN = True` for a fast smoke test, then switch to `False` for full training.

In [ ]:
import json
import shlex
import subprocess
import sys
import zipfile
from collections import Counter
from pathlib import Path

ROOT = Path(".").resolve()
if not (ROOT / "probing").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")

In [ ]:
# Core configuration
MODEL_NAME = "roberta-large-mnli"
DEVICE = "auto"

PAIRS_ZIP = ROOT / "data" / "processed" / "pairs.zip"
PAIRS_JSONL = ROOT / "data" / "processed" / "pairs.jsonl"

# Training output
OUTPUT_DIR = ROOT / "artifacts" / "finetune" / "roberta-large-mnli-spr-bin"

# Quick smoke test vs full run
QUICK_RUN = True
SMOKE_MAX_TRAIN = 2000
SMOKE_MAX_EVAL = 500
SMOKE_EPOCHS = 1.0

# Full-run defaults
FULL_EPOCHS = 3.0
TRAIN_BATCH = 16
EVAL_BATCH = 32
MAX_LEN = 256
LR = 2e-5

print("Model:", MODEL_NAME)
print("Input zip:", PAIRS_ZIP)
print("Pairs JSONL:", PAIRS_JSONL)
print("Output:", OUTPUT_DIR)
print("QUICK_RUN:", QUICK_RUN)

In [ ]:
# Extract pairs.jsonl from pairs.zip if needed
if not PAIRS_JSONL.exists():
    if not PAIRS_ZIP.exists():
        raise FileNotFoundError(f"Missing input zip: {PAIRS_ZIP}")

    with zipfile.ZipFile(PAIRS_ZIP) as zf:
        members = zf.namelist()
        jsonl_members = [m for m in members if m.endswith(".jsonl")]
        if not jsonl_members:
            raise RuntimeError(f"No .jsonl file found in {PAIRS_ZIP}")
        member = jsonl_members[0]
        PAIRS_JSONL.parent.mkdir(parents=True, exist_ok=True)
        with zf.open(member) as src, PAIRS_JSONL.open("wb") as dst:
            dst.write(src.read())

print(f"Ready: {PAIRS_JSONL} ({PAIRS_JSONL.stat().st_size / (1024**2):.1f} MB)")

In [ ]:
# Quick data sanity for the selected mapping/filtering
split_counts = Counter()
mapped_counts = Counter()
raw_label_counts = Counter()
rows_total = 0
rows_kept = 0

with PAIRS_JSONL.open(encoding="utf-8") as fh:
    for line in fh:
        rec = json.loads(line)
        rows_total += 1

        split = rec.get("split", "")
        split_counts[split] += 1

        label = int(rec["label"])
        raw_label_counts[label] += 1

        applicable = rec.get("applicable", True)
        if isinstance(applicable, str):
            applicable = applicable.strip().lower() == "true"

        if not applicable:
            continue

        rows_kept += 1
        mapped_counts["entailment" if label >= 4 else "not_entailment"] += 1

print("Total rows:", rows_total)
print("Rows kept (applicable only):", rows_kept)
print("Split counts:", dict(split_counts))
print("Raw label counts:", dict(sorted(raw_label_counts.items())))
print("Mapped label counts:", dict(mapped_counts))

In [ ]:
# Launch finetuning
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    "-m",
    "probing.finetune",
    "--model", MODEL_NAME,
    "--input", str(PAIRS_JSONL),
    "--output-dir", str(OUTPUT_DIR),
    "--train-split", "train",
    "--eval-split", "dev",
    "--label-threshold", "4",
    "--max-length", str(MAX_LEN),
    "--per-device-train-batch-size", str(TRAIN_BATCH),
    "--per-device-eval-batch-size", str(EVAL_BATCH),
    "--learning-rate", str(LR),
    "--device", DEVICE,
]

if QUICK_RUN:
    cmd += [
        "--num-train-epochs", str(SMOKE_EPOCHS),
        "--max-train-examples", str(SMOKE_MAX_TRAIN),
        "--max-eval-examples", str(SMOKE_MAX_EVAL),
    ]
else:
    cmd += ["--num-train-epochs", str(FULL_EPOCHS)]

print("Running command:
")
print(" ".join(shlex.quote(x) for x in cmd))
print()

subprocess.run(cmd, cwd=ROOT, check=True)

In [ ]:
# Inspect training metadata + metrics
meta_path = OUTPUT_DIR / "finetune_metadata.json"
if not meta_path.exists():
    raise FileNotFoundError(f"Missing metadata file: {meta_path}")

meta = json.loads(meta_path.read_text(encoding="utf-8"))

print("Saved model:", OUTPUT_DIR)
print("Train examples:", meta["data"]["train_examples"])
print("Eval examples:", meta["data"]["eval_examples"])
print("Train label counts:", meta["data"]["train_label_counts"])
print("Eval label counts:", meta["data"]["eval_label_counts"])

print("
Metrics:")
for k, v in sorted(meta.get("metrics", {}).items()):
    if isinstance(v, (int, float)):
        print(f"- {k}: {v:.6f}")
    else:
        print(f"- {k}: {v}")

In [ ]:
# Quick inference sanity check with the finetuned checkpoint
from probing.prober import Prober

sample = None
with PAIRS_JSONL.open(encoding="utf-8") as fh:
    for line in fh:
        rec = json.loads(line)
        if rec.get("split") == "dev":
            applicable = rec.get("applicable", True)
            if isinstance(applicable, str):
                applicable = applicable.strip().lower() == "true"
            if applicable:
                sample = rec
                break

if sample is None:
    raise RuntimeError("No dev/applicable sample found")

ft_prober = Prober(str(OUTPUT_DIR), device=DEVICE, threshold=0.5)
pred = ft_prober.predict_one(sample["target_text"], sample["hypothesis"], return_inputs=True)

print("Sample property:", sample.get("property"))
print("Gold label:", sample.get("label"), "(mapped entailment if >=4)")
print("Prediction:", pred["pred_label"])
print("p_entail:", pred["p_entail"])
print("p_not_entail:", pred["p_not_entail"])